# FairTickets — 02. Ecommerce Clickstream Human Funnel Baseline (Kaggle)

This notebook extracts real human **view → cart/hold → checkout vs. abandon** timing distributions from **Kaggle Ecommerce Clickstream / Website Logs** and exports `human-baseline.json` directly into `security/src/human-baseline.json` for FairTickets' **Unsupervised Human-Baseline Anomaly Detector** (`UnsupervisedBaselineDetector`).

### Workflow
1. Install dependencies (`pandas`, `numpy`, `kagglehub`).
2. Load Kaggle Ecommerce Clickstream CSV (`event_time`, `event_type`, `user_session`) or upload a CSV.
3. Compute session funnel timings (`queueToReserveMs`, `reserveToCheckoutMs`, `interEventMeanMs`, `interEventCv`, `eventsPerMinute`, `abandonmentRatio`).
4. Trim extreme automated outliers (top 2% velocity) and export Welford-compatible `human-baseline.json`.

In [ ]:
!pip -q install pandas numpy kagglehub

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd

UNSUPERVISED_FEATURES = [
    "queueToReserveMs",
    "reserveToCheckoutMs",
    "interEventMeanMs",
    "interEventCv",
    "eventsPerMinute",
    "abandonmentRatio",
]

### Step 1: Load Kaggle Clickstream CSV
Set `KAGGLE_CSV_PATH` (or upload a file in Colab). If none is provided, a small clickstream-schema sample is generated so you can test the export.

In [ ]:
KAGGLE_CSV_PATH = Path("/content/ecommerce_clickstream.csv")

if not KAGGLE_CSV_PATH.exists():
    try:
        from google.colab import files
        print("Upload Kaggle Ecommerce Clickstream CSV (columns: event_time, event_type, user_session):")
        uploaded = files.upload()
        if uploaded:
            KAGGLE_CSV_PATH = Path(next(iter(uploaded.keys())))
    except Exception:
        pass

if KAGGLE_CSV_PATH.exists():
    df = pd.read_csv(KAGGLE_CSV_PATH, usecols=lambda c: c in {"event_time", "event_type", "user_session", "timestamp", "session_id"})
    print(f"Loaded {len(df)} clickstream events from {KAGGLE_CSV_PATH}")
else:
    print("No CSV uploaded yet — generating Kaggle-clickstream schema smoke sample.")
    rng = np.random.default_rng(7)
    rows = []
    base_ts = pd.Timestamp("2026-11-01T10:00:00Z")
    for i in range(150):
        sid = f"user_sess_{i}"
        t = base_ts + pd.Timedelta(seconds=int(i * 15))
        rows.append({"event_time": t.isoformat(), "event_type": "view", "user_session": sid})
        t = t + pd.Timedelta(milliseconds=int(rng.uniform(1200, 6000)))
        rows.append({"event_time": t.isoformat(), "event_type": "cart", "user_session": sid})
        if rng.random() > 0.08:
            t = t + pd.Timedelta(milliseconds=int(rng.uniform(1500, 7500)))
            rows.append({"event_time": t.isoformat(), "event_type": "purchase", "user_session": sid})
    df = pd.DataFrame(rows)

df.head()

### Step 2: Extract Human Funnel Features & Fit Welford Baseline

In [ ]:
sess_col = "user_session" if "user_session" in df.columns else "session_id"
ts_col = "event_time" if "event_time" in df.columns else "timestamp"
type_col = "event_type" if "event_type" in df.columns else "action"

df = df.copy()
df["_ts_ms"] = (pd.to_datetime(df[ts_col], utc=True, errors="coerce").astype("int64") // 1_000_000)
df["_etype"] = df[type_col].astype(str).str.lower()
df = df.dropna(subset=[sess_col, "_ts_ms"]).sort_values([sess_col, "_ts_ms"])

session_rows = []
for _, grp in df.groupby(sess_col):
    if len(grp) < 2:
        continue
    ts = grp["_ts_ms"].to_numpy(dtype=np.float64)
    etypes = grp["_etype"].tolist()
    deltas = np.maximum(1.0, np.diff(ts))
    mean_ms = float(np.mean(deltas))
    cv = float(np.std(deltas) / mean_ms) if (len(deltas) >= 2 and mean_ms > 0) else (0.05 if mean_ms < 500 else 0.3)
    elapsed_min = max(0.01, float(ts[-1] - ts[0]) / 60_000.0)
    epm = min(240.0, float(len(ts)) / elapsed_min)

    first_view = next((t for t, e in zip(ts, etypes) if e == "view"), ts[0])
    first_cart = next((t for t, e in zip(ts, etypes) if e == "cart"), None)
    first_purch = next((t for t, e in zip(ts, etypes) if e == "purchase"), None)

    q2r = min(30_000.0, max(0.0, first_cart - first_view)) if first_cart is not None else min(30_000.0, mean_ms)
    r2c = min(30_000.0, max(0.0, first_purch - first_cart)) if (first_cart is not None and first_purch is not None) else min(30_000.0, mean_ms)

    carts = sum(1 for e in etypes if e == "cart")
    purchases = sum(1 for e in etypes if e == "purchase")
    aband = max(0.0, min(1.0, (carts - purchases) / carts)) if carts > 0 else 0.0

    session_rows.append({
        "queueToReserveMs": q2r,
        "reserveToCheckoutMs": r2c,
        "interEventMeanMs": min(30_000.0, mean_ms),
        "interEventCv": min(2.0, cv),
        "eventsPerMinute": epm,
        "abandonmentRatio": aband,
    })

base_df = pd.DataFrame(session_rows)
# Trim top 2% extreme velocity outliers so scraper noise in public logs doesn't skew human baseline
if len(base_df) > 20:
    cap = base_df["eventsPerMinute"].quantile(0.98)
    base_df = base_df[base_df["eventsPerMinute"] <= cap]

print(f"Calibrated on {len(base_df)} clean human clickstream sessions:")
print(base_df.describe().loc[["mean", "std"]])

### Step 3: Export `human-baseline.json` for FairTickets (`security/src/human-baseline.json`)

In [ ]:
n = int(len(base_df))
means, m2, stds = {}, {}, {}
for feat in UNSUPERVISED_FEATURES:
    vals = base_df[feat].to_numpy(dtype=np.float64)
    m = float(np.mean(vals))
    sum_sq = float(np.sum((vals - m) ** 2))
    s = float(max(1e-4, np.sqrt(sum_sq / max(1, n - 1))))
    means[feat] = round(m, 6)
    m2[feat] = round(sum_sq, 6)
    stds[feat] = round(s, 6)

baseline_artifact = {
    "source": "kaggle_ecommerce_clickstream_colab",
    "sampleCount": n,
    "means": means,
    "m2": m2,
    "stds": stds,
    "updatedAt": datetime.now(timezone.utc).isoformat(),
}

out_path = Path("human-baseline.json")
out_path.write_text(json.dumps(baseline_artifact, indent=2) + "\n", encoding="utf-8")
print("Exported:", out_path.resolve())
try:
    from google.colab import files
    files.download(str(out_path))
except Exception:
    pass